In [ ]:
# build tokenizer.json


import json 

# 1. Load dataset
with open("train-tinyStories-10Mb.txt", "r", encoding="utf-8") as f:
    dataset = f.read()
dataset[:1000]


# 2. Extract special token & clean training text
SPECIAL_TOKEN = "<|endoftext|>"
training_text = dataset.replace(SPECIAL_TOKEN, "")


# Find all unique characters in the clean text
unique_chars = sorted(list(set(training_text)))
unique_chars


# Create base vocabulary (ID -> Char string)
vocab = {i: char for i, char in enumerate(unique_chars)}
char_to_id = {char: i for i, char in vocab.items()}


# Assign a fixed ID for our special token at the end
special_token_id = len(vocab)
vocab[special_token_id] = SPECIAL_TOKEN


# 3. Convert training text to initial token IDs
ids = [char_to_id[c] for c in training_text]


# 4. BPE Loop (Finding merges)
vocab_size = 2**11
num_merges = vocab_size - len(vocab)
merges = {}

def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

current_new_id = len(vocab) # Start assigning IDs after special tokens

for i in range(num_merges):
    stats = get_stats(ids)
    if not stats:
        break
    best = max(stats, key=stats.get)
    
    merges[best] = current_new_id
    vocab[current_new_id] = vocab[best[0]] + vocab[best[1]]
    ids = merge(ids, best, current_new_id)
    current_new_id += 1


# 5. Save everything to a single JSON 
serializable_merges = {f"{k[0]},{k[1]}": v for k, v in merges.items()}
tokenizer_data = {
    "special_tokens": {SPECIAL_TOKEN: special_token_id},
    "vocab": {str(k): v for k, v in vocab.items()},
    "merges": serializable_merges
}

with open(f"{vocab_size}-tokenizer.json", "w", encoding="utf-8") as f:
    json.dump(tokenizer_data, f, ensure_ascii=False, indent=4)

print(f"Training finished! Saved {vocab_size}-tokenizer.json")

#TODO: build syntax merge tree, show effect of vocab_size on depth 